# 🇨🇲 Cameroon SmartTrip — Google Colab

## Projet 03 — Prototype V3

Ce notebook permet d'exécuter **tout le moteur Cameroon SmartTrip dans Google Colab**.

Il utilise :
- les données réelles du projet ;
- SQLite ;
- un parseur de langage naturel ;
- une recherche sémantique ;
- un moteur d'optimisation budgétaire ;
- un générateur d'itinéraire.

> Les prix sont datés et doivent toujours être revérifiés avant réservation.


## 1. Importer et extraire le projet

In [ ]:
from google.colab import files
import zipfile, os, shutil
from pathlib import Path

print("Importez le fichier : Projet_03_Cameroon_SmartTrip_Colab_Complet.zip")
uploaded = files.upload()

zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Aucun fichier ZIP n'a été importé.")

extract_root = Path("/content/smarttrip_project")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_root)

# Trouver automatiquement le dossier du projet
candidates = [p for p in extract_root.rglob("Cameroon-SmartTrip-Colab") if p.is_dir()]
if candidates:
    PROJECT_DIR = candidates[0]
else:
    dirs = [p for p in extract_root.iterdir() if p.is_dir()]
    PROJECT_DIR = dirs[0] if dirs else extract_root

print("Projet extrait dans :", PROJECT_DIR)


## 2. Installer les dépendances

In [ ]:
import subprocess, sys

requirements = PROJECT_DIR / "requirements_colab.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])

print("Dépendances installées.")


## 3. Vérifier les fichiers et la base SQLite

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import sys

APP_DIR = PROJECT_DIR / "app"
DB_PATH = PROJECT_DIR / "data" / "database" / "cameroon_smarttrip.db"

sys.path.insert(0, str(APP_DIR))

print("Base :", DB_PATH)
print("Existe :", DB_PATH.exists())

con = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name",
    con
)

display(tables)


## 4. Résumé des données réelles

In [ ]:
summary = {}

for table in ["hotels", "transport", "activities", "attractions", "restaurants", "data_sources"]:
    try:
        summary[table] = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table}", con)["n"].iloc[0]
    except Exception:
        summary[table] = None

pd.DataFrame({
    "Dataset": list(summary.keys()),
    "Nombre de lignes": list(summary.values())
})


## 5. Voir quelques hôtels et tarifs

In [ ]:
hotels = pd.read_sql_query(
    '''
    SELECT
        name,
        price_from_xaf,
        price_confidence,
        beach_access,
        pool,
        wifi,
        restaurant,
        verified_on
    FROM vw_budget_hotels
    ORDER BY CAST(price_from_xaf AS REAL)
    ''',
    con
)

display(hotels.head(15))


## 6. Tester la compréhension du langage naturel

In [ ]:
from nlp_parser import parse_trip_request

demande = (
    "Nous sommes un couple avec deux enfants, budget 300 000 FCFA, "
    "3 jours à Kribi depuis Douala. Je veux un hôtel calme avec piscine "
    "près de la plage et Wi-Fi, et je veux voir les chutes de la Lobé."
)

parsed = parse_trip_request(demande)
parsed.to_dict()


## 7. Tester la recherche sémantique

In [ ]:
from semantic_search import HotelSemanticSearch

search_engine = HotelSemanticSearch()

print("Moteur utilisé :", search_engine.backend)

query = "hôtel familial calme près de la plage avec piscine et wifi"

results = search_engine.search(query, top_k=10)

display(
    results[
        ["name", "price_from_xaf", "semantic_score", "beach_access", "pool", "wifi"]
    ]
)


### Optionnel — Activer les embeddings multilingues

Le projet fonctionne déjà sans API avec TF-IDF.

Pour tester de vrais embeddings multilingues dans Colab :

```python
!pip install -q sentence-transformers
```

Puis relancez la cellule précédente.


## 8. Lancer l'agent de planification

In [ ]:
from planner_agent import optimize_trip

demande = (
    "Nous sommes un couple avec deux enfants, budget 300 000 FCFA, "
    "3 jours à Kribi depuis Douala. "
    "Je veux un hôtel calme avec piscine près de la plage et Wi-Fi, "
    "et je veux voir les chutes de la Lobé."
)

plan = optimize_trip(
    demande,
    meal_budget_override=40000
)

plan


## 9. Afficher le plan de manière lisible

In [ ]:
if plan.get("status") in ["ok", "over_budget"]:
    print("🏨 Hôtel :", plan["hotel"])
    print("🚌 Transport :", plan["transport_class"])
    print("💰 Budget total :", f"{plan['request']['budget_fcfa']:,.0f} FCFA")
    print("💵 Coût calculé :", f"{plan['total_fcfa']:,.0f} FCFA")
    print("🟢 Reste :", f"{plan['remaining_fcfa']:,.0f} FCFA")
    print("🧠 Backend sémantique :", plan["semantic_backend"])
    print()

    if plan["adjustments"]:
        print("Ajustements de l'agent :")
        for x in plan["adjustments"]:
            print(" -", x)
else:
    print(plan)


## 10. Générer l'itinéraire jour par jour

In [ ]:
from itinerary import generate_itinerary, itinerary_markdown

program = generate_itinerary(plan)

for day in program:
    print(f"\nJOUR {day['day']} — {day['title']}")
    print("Matin :", day["morning"])
    print("Après-midi :", day["afternoon"])
    print("Soir :", day["evening"])


## 11. Télécharger l'itinéraire

In [ ]:
from google.colab import files

itinerary_file = Path("/content/cameroon_smarttrip_itineraire.md")
itinerary_file.write_text(itinerary_markdown(program), encoding="utf-8")

files.download(str(itinerary_file))


## 12. Afficher les meilleures alternatives

In [ ]:
alternatives = pd.DataFrame(plan.get("alternatives", []))

if not alternatives.empty:
    cols = [
        "hotel",
        "transport_class",
        "hotel_price_night",
        "total_fcfa",
        "remaining_fcfa",
        "combined_score",
        "confidence",
    ]
    display(alternatives[cols].head(10))


## 13. Tester votre propre demande

In [ ]:
ma_demande = (
    "Je pars de Yaoundé pour Kribi, nous sommes 2 personnes, "
    "2 nuits, budget 220000 CFA, je veux un hôtel près de la plage "
    "avec restaurant et pas trop cher."
)

mon_plan = optimize_trip(
    ma_demande,
    meal_budget_override=30000
)

print("Status :", mon_plan["status"])

if mon_plan.get("hotel"):
    print("Hôtel :", mon_plan["hotel"])
    print("Transport :", mon_plan["transport_class"])
    print("Total :", f"{mon_plan['total_fcfa']:,.0f} FCFA")
    print("Reste :", f"{mon_plan['remaining_fcfa']:,.0f} FCFA")


## 14. Fonction pratique pour réutiliser SmartTrip

In [ ]:
def planifier_voyage(texte, budget_repas=0):
    result = optimize_trip(texte, meal_budget_override=budget_repas)

    if not result.get("hotel"):
        return result

    print("=" * 70)
    print("🇨🇲 CAMEROON SMARTTRIP")
    print("=" * 70)
    print("Hôtel :", result["hotel"])
    print("Transport :", result["transport_class"])
    print("Coût total :", f"{result['total_fcfa']:,.0f} FCFA")
    print("Reste budget :", f"{result['remaining_fcfa']:,.0f} FCFA")
    print("Moteur :", result["semantic_backend"])

    print("\nITINÉRAIRE")
    for day in generate_itinerary(result):
        print(f"\nJour {day['day']} — {day['title']}")
        print(" Matin :", day["morning"])
        print(" Après-midi :", day["afternoon"])
        print(" Soir :", day["evening"])

    return result


## 15. Exemple final

In [ ]:
resultat = planifier_voyage(
    "Nous sommes 4 personnes, budget 350000 FCFA, "
    "3 nuits à Kribi depuis Douala. "
    "Je veux plage, piscine, restaurant et une activité aux chutes de la Lobé.",
    budget_repas=50000
)


# ✅ Fin

À ce stade, le moteur complet fonctionne dans Colab.

La suite du projet peut ajouter :
- carte interactive ;
- géocodage ;
- historique utilisateurs ;
- WhatsApp prestataires ;
- demande de réservation ;
- interface publique.
